# ACC102 Mini Assignment — Distinction Upgrade
## Analysis of Factors Affecting Engagement with AIGC-related Notes on Xiaohongshu (Red Note)

**Dataset:** `Posts.csv`  
**Research question:** Which factors are associated with higher engagement in AIGC-related Xiaohongshu posts?  
**Scope:** category, post type, tagging strategy, timing, and text-length features  
**Analytical upgrade:** descriptive analysis + statistical testing + lightweight benchmarking

## 1. Setup and data loading
This notebook is designed to run from top to bottom without manual edits. It loads the dataset, cleans key fields, engineers analysis variables, and then evaluates the main engagement drivers.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

def parse_count(val):
    val = str(val).strip()
    if val.lower() in {'nan', 'none', ''}:
        return np.nan
    if '万' in val:
        try:
            return float(val.replace('万', '')) * 10000
        except Exception:
            return np.nan
    try:
        return float(val)
    except Exception:
        return np.nan


def count_tags(x):
    if pd.isna(x):
        return 0
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return 0
    return len([t for t in s.split(',') if str(t).strip()])


df = pd.read_csv('Posts.csv', encoding='gbk')
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
df.head(3)

Loaded 1,582 rows and 21 columns.


,feature,note_id,type,title,desc,video_url,time,last_update_time,user_id,nickname,avatar,liked_count,collected_count,comment_count,share_count,ip_location,image_list,tag_list,last_modify_ts,note_url,source_keyword
0,ai-advertisement,611bab7600000000210376ef,video,Ai放射人像，15秒钟搞定！创意海报赶紧做起来吧！！??#Ai教程 #海报,Ai放射人像，15秒钟搞定！创意海报赶紧做起来吧！！??#Ai教程 #海报,http://sns-video-bd.xhscdn.com/eff2836d-d8c1-3...,1629203318000,1629203319000,5c6cd99d0000000012038d49,设会一姐,https://sns-avatar-qc.xhscdn.com/avatar/5f3dcb...,5976,5601,35,366,NaN,http://sns-webpic-qc.xhscdn.com/202410151433/1...,NaN,1728973989648,https://www.xiaohongshu.com/explore/611bab7600...,AI设计海报，广告
1,ai-advertisement,6139c042000000000102b4dc,normal,?海报思路教程：牛角包??,??这不仅是牛角包的制作教程，也是我作图思路的总结概述。\n\t\n??第一次出教程，不太细...,NaN,1631174722000,1631174722000,60e1a633000000002002b212,嘻嘻,https://sns-avatar-qc.xhscdn.com/avatar/1040g2...,5227,3291,100,251,NaN,http://sns-webpic-qc.xhscdn.com/202410151433/6...,"海报设计,原创海报设计,设计练习,海报教程",1728973989696,https://www.xiaohongshu.com/explore/6139c04200...,AI设计海报，广告
2,ai-advertisement,61c3e0d1000000000102f82c,video,5步制作炫酷破碎效果,#平面设计[话题]#??#海报设计[话题]#??#ai[话题]#??#创意#制作[话题]#?...,http://sns-video-bd.xhscdn.com/spectrum/58beae...,1640227025000,1640227026000,5cbda32c0000000012017ef0,付顽童小雅,https://sns-avatar-qc.xhscdn.com/avatar/5cbda3...,2.2万,2万,94,1611,NaN,http://sns-webpic-qc.xhscdn.com/202410151432/b...,"平面设计,海报设计,ai,创意制作,视觉设计,AI教程,碎片",1728973989548,https://www.xiaohongshu.com/explore/61c3e0d100...,AI设计海报，广告


## 2. Cleaning and feature engineering
The raw file needs two important fixes before analysis:
1. engagement counts must be converted from Chinese compact strings such as `2.2万` into numeric values;
2. time and tag fields need to be transformed into usable analytical variables.

In [2]:
df['liked_count_n'] = df['liked_count'].apply(parse_count)
df['collected_count_n'] = df['collected_count'].apply(parse_count)
df['comment_count_n'] = pd.to_numeric(df['comment_count'], errors='coerce')
df['share_count_n'] = pd.to_numeric(df['share_count'], errors='coerce')

df['engagement'] = df[['liked_count_n', 'collected_count_n', 'comment_count_n', 'share_count_n']].fillna(0).sum(axis=1)
df['post_date'] = pd.to_datetime(df['time'], unit='ms', errors='coerce')
df['year'] = df['post_date'].dt.year
df['month'] = df['post_date'].dt.month
df['weekday'] = df['post_date'].dt.day_name()
df['hour'] = df['post_date'].dt.hour

df['title'] = df['title'].fillna('')
df['desc'] = df['desc'].fillna('')
df['tag_count'] = df['tag_list'].apply(count_tags)
df['title_len'] = df['title'].astype(str).str.len()
df['desc_len'] = df['desc'].astype(str).str.len()
df['is_video'] = (df['type'] == 'video').astype(int)
df['log_engagement'] = np.log1p(df['engagement'])

df[['feature','type','liked_count','liked_count_n','collected_count','collected_count_n','engagement','tag_count','year']].head()

,feature,type,liked_count,liked_count_n,collected_count,collected_count_n,engagement,tag_count,year
0,ai-advertisement,video,5976,5976.0,5601,5601.0,11978.0,0,2021
1,ai-advertisement,normal,5227,5227.0,3291,3291.0,8869.0,4,2021
2,ai-advertisement,video,2.2万,22000.0,2万,20000.0,43705.0,7,2021
3,ai-advertisement,normal,5755,5755.0,4297,4297.0,10544.0,13,2022
4,ai-advertisement,video,8595,8595.0,8660,8660.0,17761.0,15,2022


In [3]:
missing = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(1)
}).sort_values('missing_pct', ascending=False)
missing.head(10)

,missing_values,missing_pct
ip_location,1345,85.0
video_url,1128,71.3
tag_list,93,5.9
share_count_n,4,0.3
comment_count_n,2,0.1
type,0,0.0
note_id,0,0.0
feature,0,0.0
title,0,0.0
nickname,0,0.0


The missingness check shows that some fields are sparse, but the core variables required for the assignment are sufficiently populated after cleaning. This means the analysis can focus on post characteristics without dropping most of the dataset.

## 3. Descriptive overview
Before testing specific drivers, it is useful to understand what the dataset contains in terms of categories, post formats, and time coverage.

In [4]:
print('Category distribution:')
print(df['feature'].value_counts())
print('\nPost-type distribution:')
print(df['type'].value_counts())
print('\nPosts by year:')
print(df['year'].value_counts().sort_index())

Category distribution:
feature
ai-fashion          299
ai-advertisement    256
ai-car              255
ai-food             219
ai-literature       156
ai-sport            153
ai-technology       132
ai-print            112
Name: count, dtype: int64

Post-type distribution:
type
normal    1128
video      454
Name: count, dtype: int64

Posts by year:
year
2018       1
2019       1
2020       2
2021      16
2022      36
2023     418
2024    1108
Name: count, dtype: int64


In [5]:
overview = pd.DataFrame({
    'rows': [len(df)],
    'categories': [df['feature'].nunique()],
    'year_min': [int(df['year'].min())],
    'year_max': [int(df['year'].max())],
    'median_engagement': [df['engagement'].median()],
    'median_likes': [df['liked_count_n'].median()],
    'video_share_pct': [round(df['type'].eq('video').mean() * 100, 1)],
})
overview

,rows,categories,year_min,year_max,median_engagement,median_likes,video_share_pct
0,1582,8,2018,2024,804.0,376.0,28.7


## 4. Driver 1 — Category differences
Category is likely to matter because different forms of AIGC content fit the Xiaohongshu audience differently. The chart below compares the **median engagement** of each category so that extreme viral posts do not dominate the story.

In [6]:
cat_med = df.groupby('feature')['engagement'].median().sort_values(ascending=True)
labels = [x.replace('ai-','').capitalize() for x in cat_med.index]

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(labels, cat_med.values)
ax.set_title('Figure 1. Median engagement by AIGC category')
ax.set_xlabel('Median engagement')
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, cat_med.values):
    ax.text(val + cat_med.max()*0.01, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', va='center', fontsize=9)
plt.show()

best = cat_med.idxmax().replace('ai-','').capitalize()
worst = cat_med.idxmin().replace('ai-','').capitalize()
ratio = cat_med.max() / max(cat_med.min(), 1)
print(f'{best} has the highest typical engagement, about {ratio:.1f}× the {worst} category.')

Print has the highest typical engagement, about 129.1× the Car category.


The category differences are large, which suggests that engagement is not evenly distributed across AIGC use cases. This immediately makes category one of the most important variables for strategic interpretation.

## 5. Driver 2 — Video versus normal posts
The next question is whether format matters. Here I compare video posts with normal posts using both descriptive statistics and a non-parametric significance test.

In [7]:
ptype = df.groupby('type')['engagement'].median().reindex(['normal','video'])

fig, ax = plt.subplots(figsize=(7,4.5))
ax.bar(['Normal','Video'], ptype.values)
ax.set_title('Figure 2. Median engagement by post type')
ax.set_ylabel('Median engagement')
ax.spines[['top','right']].set_visible(False)
for i, val in enumerate(ptype.values):
    ax.text(i, val + ptype.max()*0.03, f'{val:,.0f}', ha='center')
plt.show()

video = df.loc[df['type'] == 'video', 'engagement'].dropna()
normal = df.loc[df['type'] == 'normal', 'engagement'].dropna()
mann = stats.mannwhitneyu(video, normal, alternative='two-sided')
print(f"Median video engagement: {ptype['video']:,.0f}")
print(f"Median normal engagement: {ptype['normal']:,.0f}")
print(f"Video/normal ratio: {ptype['video']/ptype['normal']:.2f}x")
print(f"Mann–Whitney U p-value: {mann.pvalue:.4g}")

Median video engagement: 2,050
Median normal engagement: 595
Video/normal ratio: 3.45x
Mann–Whitney U p-value: 6.821e-14


The difference is not only large in magnitude but also statistically persuasive within the sample. This strengthens the claim that creators should take post format seriously rather than treating it as a cosmetic choice.

## 6. Driver 3 — Tagging strategy
A common creator question is whether adding more tags reliably improves discoverability. To test that, I grouped tag counts into practical buckets and compared their typical engagement levels.

In [8]:
df['tag_bin'] = pd.cut(df['tag_count'], bins=[-0.1, 2, 4, 6, 8, 100], labels=['0–2','3–4','5–6','7–8','9+'])
tag_med = df.groupby('tag_bin', observed=True)['engagement'].median()
tag_n = df.groupby('tag_bin', observed=True)['engagement'].count()

fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(tag_med.index.astype(str), tag_med.values)
ax.set_title('Figure 3. Tag-count sweet spot and median engagement')
ax.set_xlabel('Tag-count bucket')
ax.set_ylabel('Median engagement')
ax.spines[['top','right']].set_visible(False)
for i, (val, n) in enumerate(zip(tag_med.values, tag_n.values)):
    ax.text(i, val + tag_med.max()*0.03, f'{val:,.0f}\n(n={n})', ha='center', fontsize=8)
plt.show()

print(f'The best-performing tag bucket is {tag_med.idxmax()} tags.')

The best-performing tag bucket is 9+ tags.


The pattern is an inverted U-shape rather than a simple “more is better” relationship. That is analytically useful because it points toward an **optimal range** instead of a maximal one.

## 7. Driver 4 — Time and trend effects
Engagement may also vary over time because of platform growth, algorithm changes, or shifts in audience interest around AIGC.

In [9]:
year_med = df.groupby('year')['engagement'].median().sort_index()

fig, ax = plt.subplots(figsize=(8.5,4.5))
ax.plot(year_med.index, year_med.values, marker='o', linewidth=2)
ax.fill_between(year_med.index, year_med.values, alpha=0.15)
ax.set_title('Figure 4. Median engagement trend over time')
ax.set_xlabel('Year')
ax.set_ylabel('Median engagement')
ax.spines[['top','right']].set_visible(False)
for x, y in zip(year_med.index, year_med.values):
    ax.text(x, y + year_med.max()*0.02, f'{y:,.0f}', ha='center', fontsize=9)
plt.show()

In [10]:
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heat = df.pivot_table(index='weekday', columns='hour', values='engagement', aggfunc='median').reindex(weekday_order)

fig, ax = plt.subplots(figsize=(9,4.8))
im = ax.imshow(heat.fillna(0).values, aspect='auto')
ax.set_title('Figure 5. Median engagement heatmap by weekday and hour')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Weekday')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns.tolist())
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index.tolist())
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

The annual trend suggests that AIGC content became much more salient over time. The weekday-hour heatmap is more exploratory, but it is useful for the interactive app because it lets a creator inspect when engagement appears most concentrated.

## 8. Supporting evidence — correlation and benchmark model
To move beyond pure description, I added two pieces of supporting evidence:
1. a correlation matrix to summarise pairwise relationships;
2. a lightweight linear benchmark on `log_engagement` using post characteristics.

In [11]:
corr_cols = ['engagement','liked_count_n','collected_count_n','comment_count_n','share_count_n','tag_count','title_len','desc_len']
corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr.values, aspect='auto')
ax.set_title('Figure 6. Correlation matrix of engagement-related features')
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

In [12]:
model_df = df[['log_engagement','is_video','tag_count','title_len','year','feature']].dropna().copy()
X = pd.get_dummies(model_df[['is_video','tag_count','title_len','year','feature']], drop_first=True)
X.insert(0, 'Intercept', 1.0)
y = model_df['log_engagement'].values
beta, *_ = np.linalg.lstsq(X.values.astype(float), y, rcond=None)
coef = pd.Series(beta, index=X.columns).sort_values(key=lambda s: s.abs(), ascending=False)
coef.head(10)

Intercept                1090.886507
feature_ai-print            2.680905
feature_ai-car             -1.699221
feature_ai-food            -1.237251
feature_ai-sport           -1.181166
feature_ai-literature       1.168386
feature_ai-technology      -0.985752
is_video                    0.754241
year                       -0.535854
feature_ai-fashion         -0.355532
dtype: float64

The benchmark model is intentionally simple, but it is still useful. It shows which variables remain directionally important when considered together. In this notebook, the modelling step is not used to make strong causal claims; it is used to strengthen prioritisation and interpretation.

## 9. Statistical testing across categories
Because category differences look large in the charts, I also tested whether those differences are likely to be random sample noise.

In [13]:
groups = [g['engagement'].dropna().values for _, g in df.groupby('feature') if len(g) > 0]
kw = stats.kruskal(*groups)
print(f'Kruskal–Wallis p-value across categories: {kw.pvalue:.4g}')

Kruskal–Wallis p-value across categories: 1.41e-90


The very small p-value supports the descriptive conclusion that categories perform differently in a meaningful way within this dataset.

## 10. Summary table
The summary table below consolidates the most useful benchmark statistics for the dashboard and the README.

In [14]:
summary = df.groupby('feature').agg(
    posts=('engagement', 'count'),
    median_engagement=('engagement', 'median'),
    mean_engagement=('engagement', 'mean'),
    median_likes=('liked_count_n', 'median'),
    median_saves=('collected_count_n', 'median'),
    pct_video=('is_video', 'mean'),
    median_tags=('tag_count', 'median'),
).round(2)
summary.index = [x.replace('ai-','').capitalize() for x in summary.index]
summary['pct_video'] = (summary['pct_video'] * 100).round(1)
summary.sort_values('median_engagement', ascending=False)

,posts,median_engagement,mean_engagement,median_likes,median_saves,pct_video,median_tags
Print,112,18714.0,24489.40,9266.5,8805.0,79.0,8.0
Literature,156,4099.5,7131.65,1909.5,1794.5,12.0,10.0
Advertisement,256,1402.5,3927.05,574.0,569.5,11.0,9.0
Fashion,299,961.0,5763.20,470.0,323.0,30.0,8.0
Technology,132,407.5,2271.67,178.0,129.0,22.0,7.0
Sport,153,192.0,3547.27,77.0,41.0,36.0,5.0
Food,219,155.0,6422.89,70.0,54.0,30.0,7.0
Car,255,145.0,1653.14,59.0,51.0,31.0,7.0


## 11. Final interpretation
This notebook supports four main conclusions.

1. **Category is the strongest engagement driver** in the observed sample.
2. **Video posts outperform normal posts** by a large margin and the difference is statistically credible.
3. **Tagging has an optimal range** rather than a simple linear relationship.
4. **Engagement has risen over time**, which likely reflects growing AIGC attention and platform adaptation.

For a creator or marketer, the practical implication is clear: successful AIGC publishing on Xiaohongshu is not random. It depends on choosing a category with audience fit, presenting the content in a high-performing format, and using tags strategically rather than excessively.

## 12. Limitations
- The dataset does not include creator follower counts or other audience-size controls.
- The analysis is observational, so correlation should not be interpreted as causation.
- Scraped platform data may overrepresent content that the algorithm already surfaced.
- Results are specific to the Xiaohongshu context and time window used here.